## Packages

In [15]:
from DataProcessing.Plot import *

In [16]:
from Equipments import BNC575
from Equipments import Weeder
from Equipments import SR400
from Equipments import MCBOX
from Equipments import Andor
import numpy as np
import matplotlib.pyplot as plt
import pyvisa
import time

## Initialize

In [17]:
rm = pyvisa.ResourceManager()
print(rm.list_resources())

('TCPIP0::141.211.97.110::inst0::INSTR', 'ASRL1::INSTR', 'ASRL2::INSTR', 'ASRL3::INSTR', 'ASRL4::INSTR', 'ASRL5::INSTR', 'ASRL6::INSTR', 'ASRL7::INSTR', 'GPIB0::9::INSTR')


In [5]:
sr400 = SR400.SR400('GPIB0::23::INSTR',timeout=1000)

In [18]:
bnc575 = BNC575.BNC575("GPIB0::9::INSTR")

In [ ]:
weeder = Weeder.Weeder("ASRL1::INSTR")

In [20]:
# set the pulse sequence
#                   channel width  delay  output

delay = 0
# pulse_arrangement = [["C", 2, 0, "TTL"], # 480 EXC
#                      ["D", 1e-03,  2+5e-6, "TTL"], # DEI switch
#                      ["B", 10e-6, 0, "TTL"] # photon counter
                    # ]

# pulse_arrangement = [["C", 10e-6, 0, "TTL"], # 480 EXC
#                      ["D", 20e-6, 10e-6, "TTL"], # DEI switch
#                      # ["B", 10e-6, 10e-6, "TTL"] # photon counter
#                     ]


pulse_arrangement = [["B", 10e-6, 0, "TTL"]]
# pulse_arrangement = [["C", 9.9,0,"TTL"]]

# photon counter gate width
gate_width = 15e-6
gate_delay = 3e-6

# period of a sequence
# T = 10e-3
T = 1e-3

# number of the sequence cycle
cycle_number = 20

# frequency per motor step
freq_step = 78.5 #KHz

# motor step range
motor_step_range = np.arange(0,101)
relative_freq_range = freq_step * motor_step_range

# electrode setting
board_num = 0
channels = 10
bipolar = True
voltage_range = np.linspace(-5,5,101)

In [8]:
sr400.counter_set(count_mode = "INDEPENDENT", count_preset = 9e-04, count_period_num=cycle_number, dwell=0, sourceA="INPUT1",sourceB="INPUT2", gate_A_mode="FIXED", gate_A_delay=gate_delay,
                gate_A_width=gate_width,gate_B_mode="CW")

---------------------------------------
The photon counter is now set to:

Counting Mode: INDEPENDENT
T Input: 10MHz
T Preset: 0.0009s
Period number: 20
Dwell time: External
Counter A source: INPUT1
Counter B source: INPUT2

The gate A is set to:

Delay: 3.000E-6
Width: 15.00E-6


The gate B is set to: CW
---------------------------------------


In [21]:
bnc575.disable_all()

In [22]:
bnc575.disarm_all()
# bnc575.clock_set(period=T, mode="BURST", burstctr=cycle_number+1)
bnc575.clock_set(period=T,mode="CONTINUOUS")

# bnc575.channel_set(channel="H",width=pulse_arrangement[0][1],delay=pulse_arrangement[0][2], MUX="00001111")

for idn, pulse in enumerate(pulse_arrangement):
    bnc575.channel_set(channel=pulse[0],width=pulse[1],delay=pulse[2],AMP=pulse[3])

---------------------------------------
The global clock is set to:
Mode: NORM
Period: 1.00e-03s
Pulses number: N/A
---------------------------------------
---------------------------------------
The channel B is set to:
Width: 1.0000e-05s
Delay: 0.0000e+00s
Synchronize to: T0
Out: TTL

Output timers:
HGFEDCBA
00000010
---------------------------------------


In [23]:
bnc575.rearm_all()
bnc575.start_pulses()

In [ ]:
sig = np.mean(sr400.read_entire_counts(counter="A",length = cycle_number))
sig

In [ ]:
sr400.count_reset()

## Scanning gate delay

In [ ]:
gate_delay_list = np.linspace(0,1-50e-3,20)
gate_width = 50e-6
cycle_number = 20
sig_list = []
bnc575.disarm_all()
sr400.count_reset()
for gate_delay in gate_delay_list:
    sr400.counter_set(count_mode = "INDEPENDENT", count_preset = 1, count_period_num=cycle_number, dwell=0, sourceA="INPUT1",sourceB="INPUT2", gate_A_mode="FIXED", gate_A_delay=gate_delay,
                gate_A_width=gate_width,gate_B_mode="CW")
    bnc575.rearm_all()
    bnc575.start_pulses()
    time.sleep(cycle_number * T)
    sig = np.mean(sr400.read_entire_counts(counter="A",length = cycle_number))
    bnc575.disarm_all()
    print(sig)
    sig_list.append(sig)

In [ ]:
Plot2D(gate_delay_list, np.array([gate_width]), np.array(sig_list)[None,:], xlabel = "gate delay [s]", ylabel = "gate_width", zlabel = "counts")